<a href="https://colab.research.google.com/github/RefaelBitton/CloudCourse/blob/main/exercise/ex4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
url = 'https://data.gov.il/api/3/action/datastore_search?resource_id=053cea08-09bc-40ec-8f7a-156f0677aff3'

import requests
import pandas as pd
from ipywidgets import widgets, VBox, Output
from IPython.display import display

pd.set_option("display.max_columns", None)

response = requests.get(url)
data = response.json()
df = pd.DataFrame(data)
data_df = pd.DataFrame(data['result']['records'])
data_df.head()
output_area = Output()

# Create widgets
tozeret_nm_dropdown = widgets.Dropdown(
    options=[''] + sorted(data_df['tozeret_nm'].unique().tolist()),
    description='Tozeret:',
    style={'description_width': 'initial'}
)

kinuy_mishari_dropdown = widgets.Dropdown(
    options=[''],
    description='Kinuy Mishari:',
    style={'description_width': 'initial'}
)
# Function to update the second dropdown based on the selection of the first
def update_kinuy_mishari_options(change):
    if change['new']:  # Check if a valid option is selected
        filtered_values = data_df[data_df['tozeret_nm'] == change['new']]['kinuy_mishari'].unique()
        kinuy_mishari_dropdown.options = [''] + sorted(filtered_values)
    else:
        kinuy_mishari_dropdown.options = ['']



# Function to update the output area with the total records and unique 'ramat_gimur' values
def update_output(change=None):
    output_area.clear_output()
    selected_tozeret = tozeret_nm_dropdown.value
    selected_kinuy = kinuy_mishari_dropdown.value

    if selected_tozeret and selected_kinuy:
        filtered_df = data_df[
            (data_df['tozeret_nm'] == selected_tozeret) &
            (data_df['kinuy_mishari'] == selected_kinuy)
        ]
        total_records = len(filtered_df)
        unique_ramat_gimur = filtered_df['ramat_gimur'].unique()

        with output_area:
                print(f"Total Records: {total_records}")
                print(f"Unique Ramat Gimur: {', '.join(unique_ramat_gimur) if len(unique_ramat_gimur) > 0 else 'None'}")
    else:
        with output_area:
            print("Please select valid options for both dropdowns.")


# Observe changes in both dropdowns
tozeret_nm_dropdown.observe(update_kinuy_mishari_options, names='value')

kinuy_mishari_dropdown.observe(update_output, names='value')

# Display widgets and output area
display(VBox([tozeret_nm_dropdown, kinuy_mishari_dropdown, output_area]))



In [21]:
# Tab 1: Data Overview
tab1_content = widgets.Output()
with tab1_content:
    print("Data Overview:")
    display(data_df.describe())  # Summary statistics


# Tab 2: Raw Data
tab2_content = widgets.Output()
with tab2_content:
    print("Raw Data:")
    display(data_df)  # Full DataFrame

# Tab 3: Charts
tab3_content = widgets.Output()
with tab3_content:
    # Calculate frequency of shnat_yitzur
    year_counts = data_df['shnat_yitzur'].value_counts().sort_index()

    # Define 5-year steps starting from 1995 up to the maximum year in the data
    start_year = 1995
    max_year = int(year_counts.index.max())
    years_5_step = list(range(start_year, max_year + 1, 5))

    # Ensure we include all years from 1995 in our counts (filling missing with 0)
    year_counts_filtered = year_counts.reindex(years_5_step, fill_value=0)

    plt.figure(figsize=(12, 7))

    x_labels = [str(y) for y in year_counts_filtered.index]
    x_pos = np.arange(len(x_labels))

    # Create the bar chart
    bars = plt.bar(x_pos, year_counts_filtered.values, width=1.0, edgecolor='white', color='skyblue', alpha=0.7, label='Frequency')

    # Add the line plot going through the values
    plt.plot(x_pos, year_counts_filtered.values, color='navy', marker='o', linewidth=2, label='Trend Line')


    # Adjust y-axis to ensure the highest values aren't cut off
    plt.ylim(0, year_counts_filtered.max() * 1.15)

    plt.title('Distribution of Age')
    plt.xlabel('shnat_yitzur')
    plt.ylabel('Frequency')
    plt.xticks(x_pos, x_labels, rotation=45)
    plt.grid(axis='y', linestyle='--', alpha=0.5)
    plt.legend()
    plt.tight_layout()
    plt.show()
    plt.close()

# Create Tabs
tabs = widgets.Tab(children=[tab1_content, tab2_content, tab3_content])
tabs.set_title(0, 'Tab 1: Data Overview')
tabs.set_title(1, 'Tab 2: Raw Data')
tabs.set_title(2, 'Tab 3: Year Count')

# Display Tabs
display(tabs)